# TD 1 — Corrigé complet

**Analyse des données — L3 Économie**

Ce notebook contient la solution complète. Il est mis en ligne après la séance.

> **Note pour le chargé de TD.**
>
> Les chiffres cités ci-dessous proviennent de l'exécution du **8 septembre 2026**, base de l'indice **2025 = 100**. Eurostat renomme, rebase et discontinue régulièrement ses jeux de données : **réexécutez le notebook avant chaque séance** et reprenez les valeurs si elles ont bougé. Les commentaires, eux, tiennent.
>
> Trois pièges de structure, tous rencontrés en séance :
>
> 1. La colonne des postes ne porte **pas le même nom** dans les deux fichiers : `coicop` dans les pondérations, `coicop18` dans les indices.
> 2. Le code de l'ensemble n'est **pas le même** non plus : `CP00` dans les pondérations, `TOTAL` dans les indices.
> 3. La colonne `unit` n'existe **que** dans le fichier des indices.
>
> Ces trois écarts ne sont pas des détails de syntaxe. Ils viennent de ce que les deux fichiers relèvent de deux versions de la nomenclature, et c'est le sujet de la réponse 3.

## Partie 1 — Récupération

Le TD demande de passer par le *databrowser* et de téléverser deux CSV. Le notebook charge les mêmes données directement par l'API, ce qui garantit que tout le monde travaille sur le même fichier.

In [ ]:
import pandas as pd

BASE = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
OPT  = "?format=SDMX-CSV&compressed=false"

ponderations = pd.read_csv(BASE + "prc_hicp_inw"  + OPT)   # parts pour mille
indices      = pd.read_csv(BASE + "prc_hicp_ainr" + OPT)   # indices annuels

print("Ponderations :", ponderations.shape)
print(ponderations.columns.tolist())
print()
print("Indices      :", indices.shape)
print(indices.columns.tolist())

Sortie du 8 septembre 2026 :

```
Ponderations : (351477, 9)
['DATAFLOW', 'LAST UPDATE', 'freq', 'coicop', 'geo', 'TIME_PERIOD',
 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']

Indices      : (735741, 10)
['DATAFLOW', 'LAST UPDATE', 'freq', 'unit', 'coicop18', 'geo', 'TIME_PERIOD',
 'OBS_VALUE', 'OBS_FLAG', 'CONF_STATUS']
```

**Réponse 1.** Le fichier des pondérations contient **351 477 lignes** et 9 colonnes. Une ligne décrit **un poste COICOP, pour un pays, pour une année**. L'unité statistique est donc le triplet (poste, pays, année), et non le poste seul.

Le fichier des indices en contient **735 741** et 10 colonnes. La colonne supplémentaire est `unit` : le même triplet y apparaît **plusieurs fois**, une fois par unité de mesure (indice moyen annuel, taux de variation, etc.). L'unité statistique y est donc le quadruplet (poste, pays, année, unité).

C'est le premier point de vocabulaire de la séance 1 appliqué à un vrai fichier, et c'est la raison pour laquelle il faudra filtrer sur `unit` à la question suivante.

**Deux noms de colonnes, deux nomenclatures.** Les postes s'appellent `coicop` dans un fichier et `coicop18` dans l'autre. Ce n'est pas une inconséquence d'Eurostat : les pondérations sont publiées sous ECOICOP version 1, les indices sous la version 2, dite COICOP 2018. Retenez-le, la réponse 3 y revient.

In [ ]:
# Les deux fichiers ne nomment pas la colonne des postes de la meme facon.
# On le constate plutot que de le supposer.
COL_POSTE_PONDER = "coicop"     # prc_hicp_inw
COL_POSTE_INDICE = "coicop18"   # prc_hicp_ainr

print("postes, ponderations :", ponderations[COL_POSTE_PONDER].nunique(), "modalites")
print("postes, indices      :", indices[COL_POSTE_INDICE].nunique(), "modalites")

# Le code de l'ensemble n'est pas le meme non plus.
print()
print("ensemble dans les ponderations :", "CP00"  in set(ponderations[COL_POSTE_PONDER]))
print("ensemble dans les indices      :", "TOTAL" in set(indices[COL_POSTE_INDICE]))
print("CP00 present dans les indices  :", "CP00"  in set(indices[COL_POSTE_INDICE]))

In [ ]:
POSTES = ["CP0" + str(i) for i in range(1, 10)] + ["CP10", "CP11", "CP12"]


def extraire(df, colonne_poste, geo, annee, postes=POSTES, unite=None):
    """Renvoie un dictionnaire {poste: valeur} pour un pays et une annee."""
    d = df[(df["geo"] == geo)
           & (df["TIME_PERIOD"] == annee)
           & (df[colonne_poste].isin(postes))]
    if unite is not None:
        d = d[d["unit"] == unite]
    return dict(zip(d[colonne_poste], d["OBS_VALUE"]))


poids_2023  = extraire(ponderations, COL_POSTE_PONDER, "FR", 2023)
poids_2021  = extraire(ponderations, COL_POSTE_PONDER, "FR", 2021)
indice_2023 = extraire(indices, COL_POSTE_INDICE, "FR", 2023, unite="INX_A_AVG")
indice_2021 = extraire(indices, COL_POSTE_INDICE, "FR", 2021, unite="INX_A_AVG")

print("poids 2023 :", len(poids_2023), " poids 2021 :", len(poids_2021))
print("indice 2023:", len(indice_2023), " indice 2021:", len(indice_2021))
print("somme des poids 2023 :", round(sum(poids_2023.values()), 2))
print("somme des poids 2021 :", round(sum(poids_2021.values()), 2))

Douze postes de chaque côté, et la somme des pondérations vaut **1 000** pour chaque année. C'est le contrôle à faire systématiquement : si elle ne tombe pas sur 1 000, le filtre est mauvais, ou un poste a été compté deux fois.

Le filtre sur `unit="INX_A_AVG"` est obligatoire. Sans lui, le même poste apparaît plusieurs fois, et `dict()` ne garde silencieusement que la dernière occurrence rencontrée. Aucun message d'erreur, un dictionnaire de la bonne longueur, et des valeurs fausses.

## Partie 2 — Reconstitution de l'indice d'ensemble

In [ ]:
def indice_pondere(indices_poste, poids):
    """Moyenne des indices de poste, ponderee par les parts budgetaires."""
    numerateur = denominateur = 0
    for poste in poids:
        if poste in indices_poste:
            numerateur   += poids[poste] * indices_poste[poste]
            denominateur += poids[poste]
    return numerateur / denominateur


reconstitue = indice_pondere(indice_2023, poids_2023)

# Attention : dans le fichier des indices, l'ensemble se code TOTAL, pas CP00.
publie = extraire(indices, COL_POSTE_INDICE, "FR", 2023,
                  postes=["TOTAL"], unite="INX_A_AVG")["TOTAL"]

print("Reconstitue :", round(reconstitue, 2))
print("Publie      :", round(publie, 2))
print("Ecart       :", round(reconstitue - publie, 2), "point")
print("Ecart relatif : %.2f %%" % (100 * (reconstitue / publie - 1)))

Sortie du 8 septembre 2026 :

```
Reconstitue : 96.06
Publie      : 96.84
Ecart       : -0.78 point
Ecart relatif : -0.81 %
```

**Réponse 2.** L'écart vaut **0,78 point d'indice, soit 0,8 %**.

Ne le présentez pas comme négligeable. L'INSEE publie le taux d'inflation avec une précision annoncée du dixième de point. Un écart de huit dixièmes sur le niveau de l'indice est, à cette échelle, considérable. La formule du manuel ne redonne pas le chiffre publié, et la question est de savoir pourquoi.

**Réponse 3.** Quatre raisons, dans l'ordre où il faut les examiner. Les trois dernières sont les raisons méthodologiques classiques ; la première est propre à ces deux fichiers, et c'est celle que les étudiants doivent trouver seuls.

**1. Les deux fichiers ne relèvent pas de la même nomenclature.**

Les pondérations viennent de la branche ECOICOP version 1, où la colonne s'appelle `coicop` et compte 468 modalités. Les indices viennent de la branche COICOP 2018, où elle s'appelle `coicop18` et en compte 553. Le passage d'une version à l'autre a redécoupé plusieurs divisions.

Conséquence : **un code comme `CP12` ne désigne pas le même périmètre dans les deux fichiers**. On applique donc à l'indice d'un poste la pondération d'un poste qui n'a pas le même contenu.

> **À vérifier dans les métadonnées avant la séance**, division par division. C'est un excellent exercice de lecture de documentation, et la piste à explorer en premier : renommer la colonne fait tourner le code, mais ne fait pas coïncider les deux nomenclatures.

**2. Le chaînage.** L'indice publié est chaîné : il enchaîne des indices annuels à pondérations mobiles. Une moyenne pondérée en une seule étape n'est pas la même opération.

**3. La période de référence des pondérations.** Les pondérations d'une année sont établies à partir des dépenses d'une année antérieure, réévaluées aux prix de décembre précédent. Elles ne se rapportent donc pas à la période des indices qu'elles pondèrent.

**4. Les arrondis.** Les pondérations sont publiées en parts pour mille avec deux décimales ; l'agrégation cumule ces arrondis. Cette raison-là, seule, ne produit pas huit dixièmes de point.

Le point pédagogique : la formule du manuel est correcte, mais elle ne décrit pas ce que fait un institut statistique, et surtout elle suppose que les deux fichiers parlent de la même chose. **C'est le genre d'écart qu'il faut savoir diagnostiquer plutôt que masquer.**

### La réponse à la question de la séance

Le TD s'ouvre sur une question simple : de combien les prix ont-ils augmenté en France entre 2021 et 2023 ? Elle mérite d'être traitée explicitement, sur l'indice publié.

In [ ]:
total_2021 = extraire(indices, COL_POSTE_INDICE, "FR", 2021,
                      postes=["TOTAL"], unite="INX_A_AVG")["TOTAL"]
total_2022 = extraire(indices, COL_POSTE_INDICE, "FR", 2022,
                      postes=["TOTAL"], unite="INX_A_AVG")["TOTAL"]
total_2023 = publie

print("Indice d'ensemble France :", total_2021, total_2022, total_2023)
print("Hausse 2021 -> 2022 : %.2f %%" % (100 * (total_2022 / total_2021 - 1)))
print("Hausse 2022 -> 2023 : %.2f %%" % (100 * (total_2023 / total_2022 - 1)))
print("Hausse 2021 -> 2023 : %.2f %%" % (100 * (total_2023 / total_2021 - 1)))

```
Indice d'ensemble France : 86.54 91.65 96.84
Hausse 2021 -> 2022 : 5.90 %
Hausse 2022 -> 2023 : 5.66 %
Hausse 2021 -> 2023 : 11.90 %
```

**Les prix ont augmenté de 11,9 % en France entre 2021 et 2023.** Deux remarques à faire en séance.

Les indices valent 86,5 et 96,8, donc moins de 100, alors que les prix montent. Ce n'est pas une anomalie : la **base est 2025 = 100**, et 2021 comme 2023 sont antérieurs à la base. Un indice n'a aucun sens sans sa base, et c'est le genre de détail qui fait rendre des copies aberrantes.

5,90 % puis 5,66 % ne font pas 11,56 % mais 11,90 %. Les taux de croissance ne s'additionnent pas, ils se composent.

## Partie 3 — Le choix des pondérations

In [ ]:
avec_2023 = indice_pondere(indice_2023, poids_2023)
avec_2021 = indice_pondere(indice_2023, poids_2021)

print("Indice 2023, ponderations 2023 :", round(avec_2023, 2))
print("Indice 2023, ponderations 2021 :", round(avec_2021, 2))
print("Ecart                          :", round(avec_2021 - avec_2023, 2), "point")

```
Indice 2023, ponderations 2023 : 96.06
Indice 2023, ponderations 2021 : 96.04
Ecart                          : -0.02 point
```

**Réponse 4.** L'écart vaut **deux centièmes de point**, et il va dans le sens inverse de celui qu'on attend : ce sont les pondérations **récentes** qui donnent l'indice le plus élevé, de très peu.

> **Ne faites pas dire à ce résultat la leçon de Laspeyres.** C'est la correction la plus importante de ce corrigé, et elle porte sur deux choses.
>
> **La première est conceptuelle.** Ce calcul n'est pas une comparaison Laspeyres/Paasche. Les deux versions utilisent **les mêmes indices**, ceux de 2023 en base 2025. On ne compare pas deux mesures d'une évolution de prix entre deux dates avec les quantités de l'une ou de l'autre ; on repondère un niveau d'indice déjà calculé. Le signe de l'écart n'est donc pas garanti par la théorie.
>
> **La seconde est empirique.** Sur ces deux années précises, l'écart est de 0,02 point. Il est **plus petit que l'arrondi de publication**. Annoncer un enjeu distributif sur un écart pareil serait raconter une histoire que les données ne portent pas.

Comparez les deux ordres de grandeur rencontrés jusqu'ici : **0,78 point** pour l'écart entre la formule du manuel et l'indice publié, **0,02 point** pour le choix des pondérations. Le premier mérite une explication, le second est du bruit. C'est la comparaison que demande la question, et elle passe avant toute interprétation du signe.

L'exercice reste utile, et davantage qu'une confirmation : il montre qu'un mécanisme théorique bien établi peut être invisible sur un couple d'années donné, et il oblige à chercher pourquoi. C'est l'objet de la réponse 5.

In [ ]:
variation = {p: poids_2023[p] - poids_2021[p] for p in poids_2021 if p in poids_2023}

print("%-6s %8s %8s %10s %10s" % ("poste", "I 2021", "I 2023", "hausse %", "dW pmille"))
for p, v in sorted(variation.items(), key=lambda x: x[1]):
    hausse = 100 * (indice_2023[p] / indice_2021[p] - 1)
    print("%-6s %8.1f %8.1f %+10.1f %+10.1f"
          % (p, indice_2021[p], indice_2023[p], hausse, v))

```
poste    I 2021   I 2023   hausse %  dW pmille
CP01       80.8     97.4      +20.5      -16.0
CP02       84.4     92.1       +9.1       -9.6
CP12       82.7     86.6       +4.7       -7.9
CP04       83.3     95.9      +15.1       -7.9
CP08      115.2    112.0       -2.8       -4.7
CP05       89.3    100.4      +12.4       -2.9
CP06       91.3     93.4       +2.3       -1.2
CP03       93.6     98.8       +5.6       -0.9
CP10       87.2     93.0       +6.7       +0.4
CP09       88.7     96.2       +8.5       +0.7
CP07       86.8     99.3      +14.4      +17.8
CP11       86.2     94.8      +10.0      +32.2
```

Plutôt que de lire le tableau à l'œil, mesurons la relation que la question suppose.

In [ ]:
postes_communs = sorted(variation)
hausses = pd.Series({p: 100 * (indice_2023[p] / indice_2021[p] - 1) for p in postes_communs})
poids_d = pd.Series({p: variation[p] for p in postes_communs})

print("correlation sur les 12 postes        : %.3f" % hausses.corr(poids_d))

sans_rebond = [p for p in postes_communs if p not in ("CP07", "CP11")]
print("en retirant CP07 et CP11             : %.3f"
      % hausses[sans_rebond].corr(poids_d[sans_rebond]))

```
correlation sur les 12 postes        : -0.020
en retirant CP07 et CP11             : -0.579
```

**Réponse 5.** La question de l'énoncé est : les postes dont la pondération a le plus reculé sont-ils aussi ceux dont le prix a le plus augmenté ? **En partie seulement, et il faut le dire.**

**Ce qui va dans le sens attendu.** `CP01`, produits alimentaires : la plus forte hausse de prix du tableau, **+20,5 %**, et le plus fort recul de pondération, **−16,0 pour mille**. `CP04`, logement et énergie : +15,1 % et −7,9. Ce sont les deux postes au cœur du choc inflationniste de 2022, et la substitution s'y lit directement.

**Ce qui va contre.** `CP11`, restauration et hébergement, gagne **+32,2 pour mille**, de loin la plus forte progression, alors que ses prix montent de 10 %. `CP07`, transports, gagne +17,8 avec des prix en hausse de 14,4 %. Ces deux postes contredisent frontalement l'hypothèse.

**Pourquoi.** Parce que **2021 n'est pas une année de consommation normale.** Restaurants, hôtels et transports étaient encore largement fermés ou évités. Leur part dans le budget des ménages en 2021 est artificiellement basse, et la remontée observée en 2023 est un retour à la normale, pas une réaction aux prix relatifs.

**Le chiffre qui tranche.** La corrélation entre hausse de prix et variation de pondération vaut **−0,02** sur les douze postes : rien du tout. En retirant les deux postes de rattrapage post-Covid, elle passe à **−0,58**, le signe négatif attendu et une association nette.

Ce n'est pas un contre-exemple à la théorie. C'est un cas où un second mécanisme, plus puissant sur cette période, masque le premier. **Formulez-le comme une association conditionnelle, jamais comme un effet mesuré** : douze points ne permettent d'établir aucune causalité, et le choix de retirer deux d'entre eux est lui-même une décision à justifier.

**Réponse 6.** Le raisonnement reste juste : un ménage dont les revenus sont indexés préfère la version qui majore l'indice, l'organisme payeur préfère l'autre. Le choix d'indice est bien un enjeu distributif, et non une subtilité technique.

Mais **sur ces deux années, l'enjeu est nul**, et il faut le chiffrer plutôt que de l'affirmer.

**Le calcul.** L'écart de 0,02 point sur un indice de 96,06 fait une différence relative de **0,021 %**. Sur une pension de 1 500 euros par mois, cela représente **31 centimes par mois**, soit **3,75 euros par an**.

Exigez ce calcul. Il tient en une ligne, et il retourne la conclusion que les étudiants attendaient : le choix des pondérations est un enjeu distributif réel en principe, et sur ce couple d'années il ne vaut pas la discussion qu'on lui consacrerait.

La leçon est là, et elle vaut mieux qu'une confirmation : **on vérifie l'ordre de grandeur avant de raconter une histoire de gagnants et de perdants.** Un mécanisme réel peut être quantitativement négligeable sur la période qu'on regarde.

## Partie 4 — Comparaison internationale

In [ ]:
for pays in ["FR", "HU", "BG", "EE", "DE"]:
    p = extraire(ponderations, COL_POSTE_PONDER, pays, 2023)
    if p:
        print("%s : indices francais, ponderations %s -> %.2f"
              % (pays, pays, indice_pondere(indice_2023, p)))

```
FR : indices francais, ponderations FR -> 96.06
HU : indices francais, ponderations HU -> 96.45
BG : indices francais, ponderations BG -> 96.84
EE : indices francais, ponderations EE -> 96.81
DE : indices francais, ponderations DE -> 96.18
```

**Réponse 7.** Les mêmes prix français produisent des indices différents selon la structure de consommation appliquée. L'écart va de **96,06** avec le panier français à **96,84** avec le panier bulgare, soit **0,78 point, ou 0,8 %**.

Deux remarques à faire ressortir.

L'écart est **le plus grand avec la Bulgarie et l'Estonie**, où l'alimentation et l'énergie pèsent lourd dans le budget des ménages, et **le plus faible avec l'Allemagne**, dont la structure de consommation ressemble à la française. Le classement n'est pas aléatoire : il suit la proximité des paniers.

Cet écart de 0,8 % est **du même ordre que l'écart entre l'indice reconstitué et l'indice publié** de la partie 2. Autrement dit, changer de panier déplace l'indice autant que l'ensemble des conventions méthodologiques réunies. Ce n'est pas rien.

**Réponse 8.** L'harmonisation de l'IPCH porte sur la **méthode** : nomenclature commune, champ de couverture, traitement des soldes, des logements occupés par leur propriétaire, des prix administrés. Elle ne porte **pas sur les pondérations**, qui reflètent par construction la consommation nationale de chaque pays.

Comparer deux taux d'inflation nationaux, c'est donc comparer **deux paniers différents évalués selon des règles communes**. L'harmonisation garantit que l'écart mesuré vient bien des prix et des paniers, et non de conventions comptables divergentes.

C'est aussi la raison pour laquelle la BCE suit l'agrégat de la zone euro, construit comme une moyenne pondérée des indices nationaux, plutôt que la moyenne des taux nationaux.

> **Une réserve à signaler aux étudiants attentifs.** Ce calcul hérite du décalage de nomenclature relevé en réponse 3 : les niveaux ne sont pas exacts. La comparaison entre pays, elle, reste valable, puisque le même décalage s'applique aux cinq pays.

## Erreurs fréquentes à anticiper

| Erreur | Symptôme | Ce qu'elle enseigne |
|---|---|---|
| Chercher `coicop` dans le fichier des indices | `KeyError: 'coicop'` | Deux fichiers d'une même source ne suivent pas forcément la même nomenclature |
| Renommer `coicop18` en `coicop` et passer à la suite | Le code tourne, l'écart reste inexpliqué | Faire tourner n'est pas faire juste |
| Chercher `CP00` dans le fichier des indices | Dictionnaire vide, puis `KeyError` | Le code de l'ensemble se lit dans la documentation, il ne se devine pas |
| Ne pas filtrer sur `unit` | Dictionnaire de la bonne longueur, valeurs fausses | Un dictionnaire écrase silencieusement les doublons de clé |
| Ne pas vérifier que les poids somment à 1 000 | Dénominateur aberrant | Un contrôle d'une ligne évite une heure de débogage |
| Lire l'indice sans regarder sa base | « Les prix ont baissé, l'indice vaut 86 » | Un indice n'a de sens qu'avec sa base |
| Additionner 5,90 % et 5,66 % | 11,56 % au lieu de 11,90 % | Les taux de croissance se composent |
| Conclure Laspeyres sur un écart de 0,02 point | Une histoire que les données ne portent pas | On vérifie l'ordre de grandeur avant d'interpréter le signe |
| Dire que la substitution « explique » les pondérations | Une causalité affirmée sur douze points | Une association conditionnelle n'est pas un effet |

---

## Pour la prochaine édition du cours

Trois points à reprendre dans l'énoncé, repérés en séance par le chargé de TD.

1. L'énoncé annonce le code `CP00` pour l'ensemble. C'est vrai du fichier des pondérations, faux du fichier des indices, où il faut `TOTAL`. À corriger ou à transformer en question.
2. L'énoncé ne prévient pas que la colonne des postes change de nom entre les deux fichiers. Le laisser tel quel est défendable, à condition que le corrigé du chargé de TD le mentionne.
3. Le couple 2021–2023 est pédagogiquement piégeux pour la partie 3 : le rebond post-Covid écrase l'effet de substitution. Un couple plus ancien, ou 2022–2024, donnerait un résultat plus net. Le garder est aussi un choix défendable, et c'est celui qu'a retenu ce corrigé : un résultat qui ne sort pas comme prévu enseigne davantage qu'un résultat attendu.